# MoP + DivPO — Single-LoRA Baseline Training

This notebook trains the missing `single/all` LoRA baseline for the journal evaluation.

## What this trains

`single/all` is one LoRA adapter trained on the merged SFT data from all four personas:

- `contrarian`
- `systems_thinker`
- `cross_domain_analogist`
- `minimalist`

It is the no-MoP control baseline. If MoP is useful, the four separate persona adapters should outperform this single merged adapter on diversity/persona-fidelity metrics.

## Kaggle settings

| Setting | Value |
|---|---|
| Accelerator | GPU T4 x2 preferred; single T4/P100 should also work |
| Internet | On |
| Secret | `HF_TOKEN` = HuggingFace write token |

## Recommended run mode

Use **Save Version -> Save & Run All** so the job continues on Kaggle even if your browser disconnects.

Expected runtime: about 20-45 minutes on T4 x2, depending on Kaggle load.

---
## Cell 1 — Install Dependencies

In interactive mode, restart the kernel after this cell if Kaggle has already imported PEFT/Transformers in the session. In Save Version mode, the fresh kernel can continue.

In [ ]:
import os

INTERACTIVE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive"

!pip install -q --upgrade transformers peft trl accelerate datasets huggingface_hub sentence-transformers
!pip uninstall -y -q torchao

if INTERACTIVE:
    print("\nInteractive mode: restart the kernel once if this session previously imported PEFT/Transformers, then continue from Cell 2.")
else:
    print("Save Version mode: continuing on a fresh kernel.")

---
## Cell 2 — Credentials, Repo, and Branch

In [ ]:
import os
import sys
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

REPO_DIR = "/kaggle/working/mop-divpo-llm-counter-argument"
REPO_URL = "https://github.com/DasonTio/mop-divpo-llm-counter-argument.git"
BRANCH = "feat/eval-pipeline"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

# Force the exact remote branch that contains the evaluation pipeline.
# Notebook shell commands do not stop the cell by default, so we verify explicitly below.
!git -C {REPO_DIR} fetch origin {BRANCH} --prune
!git -C {REPO_DIR} checkout -B {BRANCH} origin/{BRANCH}

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.environ["PYTHONPATH"] = os.path.join(REPO_DIR, "src")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Working dir:", os.getcwd())
print("Branch:")
!git status --short --branch
print("HF token:", os.environ["HF_TOKEN"][:8] + "...")

required = [
    "scripts/train_single_lora.py",
    "src/mop_divpo/inference/generate.py",
    "src/mop_divpo/hub.py",
]
missing = [path for path in required if not os.path.exists(path)]
if missing:
    raise RuntimeError(
        "Kaggle checked out the wrong repository state. Missing: " + ", ".join(missing) + "\n"
        "Make sure branch feat/eval-pipeline is pushed to GitHub, then rerun Cell 2."
    )
print("Required training files found.")

---
## Cell 3 — Verify GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU detected. In Kaggle settings, set Accelerator to GPU before running training.")

---
## Cell 4 — Train and Push `single/all`

This command loads the four persona SFT datasets from HuggingFace, merges them, trains one LoRA adapter, and pushes it to `DasonTio/mop-divpo-coauthor/single/all`.

In [ ]:
!python scripts/train_single_lora.py --name all --epochs 3 --batch-size 8 --grad-accum 4 --max-seq-len 512

---
## Cell 5 — Verify `single/all` Loads from Hub

This is a fast inference smoke test. If this cell generates text, the baseline adapter is ready for Phase 3 evaluation.

In [ ]:
import os
import sys

sys.path.insert(0, "src")

from mop_divpo.inference.generate import generate

result = generate(
    prompt="Remote work is strictly better for productivity.",
    personas=["contrarian"],
    adapter_stage="single",
    token=os.environ["HF_TOKEN"],
    n=1,
    temperature=0.7,
    max_new_tokens=120,
    as_counter_argument=True,
)

print(result["contrarian"][0])

---
## After This Notebook Succeeds

Continue with Phase 1 persona distinctness:

```bash
python scripts/experiment_persona_distinctness.py --stage divpo --n 2 --limit-prompts 3
python scripts/experiment_persona_distinctness.py --stage divpo --n 5
```

Then proceed to Phase 3 generation:

```bash
python scripts/run_baseline_evaluation.py --only-generate
```